In [1]:
import pandas as pd
from collections import defaultdict


In [8]:
import pandas as pd

buggy_convos = pd.read_parquet("hf://datasets/regularpooria/buggy-conversation-redo/data/train-00000-of-00001.parquet")

/home/regularpooria/Projects/WildCode/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv("../../results/buggy_convos_all_sarif_results.csv")
opengrep_wildcode_conversation_redo = {
    model: model_df
    for model, model_df in df.groupby("model")
}
opengrep_wildcode_conversation_redo

{'command_a':                       conversation_hash      model  \
 0      011b49c282451ca4ec1eb160eabf21fc  command_a   
 1      011b49c282451ca4ec1eb160eabf21fc  command_a   
 2      011b49c282451ca4ec1eb160eabf21fc  command_a   
 3      011b49c282451ca4ec1eb160eabf21fc  command_a   
 4      011b49c282451ca4ec1eb160eabf21fc  command_a   
 ...                                 ...        ...   
 28357  e6e150a0d5b7f5ade792d70e9d8575b4  command_a   
 28358  ebc90105525ca38270bf3fff034b48d2  command_a   
 28359  ee2c439d1045ed0be22f9ba735346f29  command_a   
 28360  f65c4c09f07f017ce0dcb8dfa9e8ac06  command_a   
 28361  fb5da0ebb3a2071db051a6811a14cf80  command_a   
 
                             variant language  error_line  error_character  \
 0      act_as_a_security_researcher        c           3               14   
 1      act_as_a_security_researcher        c           4               14   
 2      act_as_a_security_researcher        c           5               14   
 3      act_a

In [9]:
wildcode_conversation_hash = set(buggy_convos["conversation_id"])

In [5]:
rules = {
    "random": {
        "java": ["weak-random"],
        "csharp": [
            "use_weak_rng_for_keygeneration",
        ],
        "javascript": [
            "JS_WEAK_RNG",
        ],
        "python": [
            "PYTHON_WEAK_RNG",
            "PYTHON_WEAK_RNG_UNQUALIFIED",
            "PYTHON_WEAK_RNG_WRAPPER",
        ],
    },
    "unsafe_memory": {
        "c": [
            "insecure-use-gets-fn",
            "insecure-use-memset",
            "insecure-use-printf-fn",
            "insecure-use-strcat-fn",
            "insecure-use-scanf-fn",
            "insecure-use-string-copy-fn",
        ]
    },
    "sql": {
        "java": [
            "tainted-sql-string",
            "tainted-sqli",
            "hibernate-sqli",
            "jdbc-sqli",
            "jdo-sqli",
            "jpa-sqli",
            "tainted-sql-from-http-request",
            "turbine-sqli",
            "vertx-sqli",
            "mongodb-nosqli",
            "tainted-sql-string",
        ],
        "csharp": [
            "csharp-sqli",
        ],
        "javascript": [
            "knex-sqli",
            "mysql-sqli",
            "pg-sqli",
            "sequelize-sqli",
            "tainted-sql-string",
            "node-knex-sqli",
            "node-mssql-sqli",
            "node-mysql-sqli",
            "node-postgres-sqli",
        ],
        "php": [
            "tainted-sql-string",
            "laravel-sql-injection",
            "wp-sql-injection-audit",
            "",
        ],
        "python": [
            "mysql-sqli",
            "psycopg-sqli",
            "pymssql-sqli",
            "pymysql-sqli",
            "sqlalchemy-sqli",
            "tainted-sql-string",
            "sql-injection-using-extra-where",
            "sql-injection-using-rawsql",
            "sql-injection-db-cursor-execute",
            "sql-injection-using-raw",
            "aiopg-sqli",
            "asyncpg-sqli",
            "pg8000-sqli",
            "pyramid-sqlalchemy-sql-injection",
            "sqlalchemy-sql-injection",
            "sqlalchemy-execute-raw-query",
            "avoid-sqlalchemy-text",
        ],
    },
    "hash": {
        "java": [
            "use-of-md5",
            "use-of-weak-rsa-key",
            "use-of-sha1",
            "use-of-rc4",
            "use-of-rc2",
            "use-of-md5-digest-utils",
            "use-of-default-aes",
            "use-of-aes-ecb",
            "use-of-blowfish",
            "rsa-no-padding",
            "no-null-cipher",
            "gcm-nonce-reuse",
            "gcm-detection",
            "ecb-cipher",
            "desede-is-deprecated",
            "des-is-deprecated",
        ],
        "csharp": [
            "use_weak_rsa_encryption_padding",
            "use_deprecated_cipher_algorithm",
            "X509Certificate2-privkey",
        ],
        "javascript": [
            "aead-no-final",
            "create-de-cipher-no-iv",
            "gcm-no-tag-length",
            "md5-used-as-password",
        ],
        "php": [
            "weak-crypto",
            "md5-used-as-password",
            "md5-loose-equality",
            "mcrypt-use",
            "openssl-decrypt-validate",
        ],
        "python": [
            "crypto-mode-without-authentication",
            "insufficient-rsa-key-size",
            "insufficient-dsa-key-size",
            "insecure-hash-algorithm-sha1",
            "insecure-hash-algorithm-md5",
            "insecure-hash-algorithm-md4",
            "insecure-hash-algorithm-md2",
            "insecure-cipher-algorithm-xor",
            "insecure-cipher-algorithm-rc4",
            "insecure-cipher-algorithm-rc2",
            "insecure-cipher-algorithm-des",
            "insecure-cipher-algorithm-blowfish",
            "insecure-hash-function",
            "insecure-hash-algorithm-sha1",
            "md5-used-as-password",
            "hashids-with-django-secret",
            "crypto-mode-without-authentication",
            "insufficient-ec-key-size",
            "insecure-cipher-mode-ecb",
            "insecure-cipher-algorithm-idea",
            "insecure-cipher-algorithm-arc4",
            "empty-aes-key",
        ],
    },
}

In [11]:
for model, model_df in opengrep_wildcode_conversation_redo.items():
    print(f"============== MODEL: {model} ==============")

    for variant, value in model_df.groupby("variant"):
        print(f"---------- VARIANT: {variant} ----------")

        conversation_hashes = set(value["conversation_hash"])
        print(
            f"VULNERABLE CONVOS: "
            f"Wildcode: {len(wildcode_conversation_hash)}, "
            f"Conversation Redo: {len(conversation_hashes)}"
        )
        print(f"Added vulnerabilities: {len(conversation_hashes - wildcode_conversation_hash)}")
        print(f"Removed vulnerabilities: {len(wildcode_conversation_hash - conversation_hashes)}")

        for rule, rule_val in rules.items():
            allowed_rules_list = [
                r for sublist in rule_val.values() for r in sublist
            ]

            # ---- wildcode aggregation ----
            language_rule_results_default = {}

            for _, row in opengrep_wildcode.iterrows():
                error_id = row["error_id"].split(".")[-1]
                if error_id not in allowed_rules_list:
                    continue

                language = row["language"]
                conversation_hash = row["conversation_hash"]

                language_rule_results_default.setdefault(
                    language, {"count": 0, "hashes": set()}
                )
                language_rule_results_default[language]["count"] += 1
                language_rule_results_default[language]["hashes"].add(conversation_hash)

            # ---- redo aggregation (variant-scoped) ----
            language_rule_results_redo = {}

            for _, row in value.iterrows():
                error_id = row["error_id"].split(".")[-1]
                if error_id not in allowed_rules_list:
                    continue

                language = row["language"]
                conversation_hash = row["conversation_hash"]

                language_rule_results_redo.setdefault(
                    language, {"count": 0, "hashes": set()}
                )
                language_rule_results_redo[language]["count"] += 1
                language_rule_results_redo[language]["hashes"].add(conversation_hash)

            print(f"RULE: {rule}")
            for language, redo_val in language_rule_results_redo.items():
                wildcode_count = language_rule_results_default.get(language, {}).get("count", 0)
                print(
                    f"    {language}, "
                    f"Wildcode: {wildcode_count}, "
                    f"Conversation Redo: {redo_val['count']}"
                )


============== MODEL: command_a ==============
---------- VARIANT: act_as_a_security_researcher ----------
VULNERABLE CONVOS: Wildcode: 819, Conversation Redo: 320
Added vulnerabilities: 0
Removed vulnerabilities: 499
RULE: random
    java, Wildcode: 29, Conversation Redo: 12
RULE: unsafe_memory
    c, Wildcode: 1807, Conversation Redo: 1464
RULE: sql
    python, Wildcode: 116, Conversation Redo: 6
RULE: hash
    java, Wildcode: 22, Conversation Redo: 24
    python, Wildcode: 78, Conversation Redo: 6
---------- VARIANT: original ----------
VULNERABLE CONVOS: Wildcode: 819, Conversation Redo: 335
Added vulnerabilities: 0
Removed vulnerabilities: 484
RULE: random
    java, Wildcode: 29, Conversation Redo: 12
RULE: unsafe_memory
    c, Wildcode: 1807, Conversation Redo: 1350
RULE: sql
    php, Wildcode: 9, Conversation Redo: 6
    python, Wildcode: 116, Conversation Redo: 6
RULE: hash
    java, Wildcode: 22, Conversation Redo: 12
    python, Wildcode: 78, Conversation Redo: 18
===========